# Speech Emotion Router (SER): Feature Extraction 2 - 2D Log-Mel Spectrogram Representation

In the previous phase ([`03_nonlinear_classifications.ipynb`](./03_nonlinear_classifications.ipynb)), our tuned non-linear RBF Support Vector Classifier established that while flexible kernels lift overall accuracy to **43.13%** and achieve **>95%** gender separation, static time-pooled summary statistics in`herently constrain emotional valence classification.

This phase transitions our feature engineering pipeline from collapsed 1D tabular vectors to continuous 2D spatial-temporal representations, preserving the acoustic dynamics required for deep learning architectures.

**Phase Objectives & Experimental Scope:**

- We will use the learnings from [`02_baseline_modeling.ipynb`](./02_baseline_model.ipynb) and apply the silence threshold (`top_db=25`) across all 1,440 raw audio trials to strip leading and trailing studio buffers while preserving active speech dynamics (~1.74s mean active duration).

- Compute Short-Time Fourier Transforms (STFT with $N_{\text{FFT}}=2048$, $\text{hop\_length}=512$) projected across 128 Mel filter banks and converted to decibels (`power_to_db`) to capture time-varying formant trajectories and pitch contours.

- **Uniform Tensor Standardization:** Standardize time dimensions into fixed 128-frame grids ($\sim3.0$ seconds) via center-cropping or floor decibel padding, producing uniform `(128, 128)` 2D tensors **optimized for convolutional pooling**.

- Conduct side-by-side spectral comparisons across contrasting emotional states (e.g., `angry` vs. `calm`) to visually confirm frequency resolution and dynamic range fidelity prior to model ingestion.

- Serialize the complete tensor array, speaker groupings, and target vectors into `data/processed/spectrograms/` to completely decouple audio I/O from 2D CNN training iterations.

In [1]:
from pathlib import Path
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [6]:
# Resolve paths
project_root = Path.cwd().resolve().parent
processed_dir = project_root / "data" / "processed"
tensors_dir = processed_dir / "spectrograms"
tensors_dir.mkdir(parents=True, exist_ok=True)

# Load the prior dataframe to adding the MFCC coeff
metadata_path = processed_dir / "01_3_df_cleaned.parquet"
df_cleaned = pd.read_parquet(metadata_path)
print(f"Loaded {len(df_cleaned)} records from {metadata_path.name}")
df_cleaned.info()

Loaded 1440 records from 01_3_df_cleaned.parquet
<class 'pandas.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   file_name     1440 non-null   str    
 1   file_path     1440 non-null   str    
 2   actor_id      1440 non-null   int64  
 3   gender        1440 non-null   str    
 4   emotion       1440 non-null   str    
 5   intensity     1440 non-null   str    
 6   statement     1440 non-null   str    
 7   repetition    1440 non-null   int64  
 8   duration_sec  1440 non-null   float64
dtypes: float64(1), int64(2), str(6)
memory usage: 392.2 KB


We don't need the MFCC Coefficients for the planned CNN. Therefore, we loaded the prior DataFrame that contains the original metadata information without them and then modify it to have the updated compound target variable `gender_emotion`

In [7]:
# Construct 16-class compound target vector
df_cleaned["target"] = df_cleaned["gender"] + "_" + df_cleaned["emotion"]

print(f"Loaded {len(df_cleaned)} records from {metadata_path.name}")
print(f"Unique classes: {df_cleaned['target'].nunique()} (Expected: 16)")
print(f"Unique actors: {df_cleaned['actor_id'].nunique()} (Expected: 24)")

Loaded 1440 records from 01_3_df_cleaned.parquet
Unique classes: 16 (Expected: 16)
Unique actors: 24 (Expected: 24)


## Extracing the Spectrogram

### Acoustic Signal Processing Justification

Transforming raw 1D audio waveforms into fixed-dimension 2D Log-Mel spectrograms is grounded in standard speech processing principles:

1. **Perceptual Frequency Scaling (Mel Scale):**
   * Human frequency perception is non-linear, with higher resolution at lower frequencies (<1 kHz) and wider critical bandwidth integration at higher frequencies (Stevens et al., 1937).
   * Mapping linear STFT bins onto a 128-band Mel filter bank emphasizes fundamental speech formants ($F_1, F_2, F_3$) that encode vocal tract geometry and affective prosody.

2. **Logarithmic Dynamic Range Compression (Decibel Scale):**
   * Perceptual loudness scales logarithmically with acoustic power ($\text{dB} = 10 \cdot \log_{10}(P / P_{\text{ref}})$).
   * Decibel compression balances subtle vocal tract consonants against energetic vowel formants into stable ranges for deep learning.

3. **Spatial-Temporal Representation for Convolution:**
   * Static summary statistics ($\mu, \sigma$) flatten dynamic pitch contours and formant trajectories.
   * Preserving time-frequency geometry as a $128 \times 128$ image retains local spectral correlations and prosodic inflections required for 2D convolutional feature learning (Huang et al., 2001; Schuller, 2018).

In [9]:
# Audio & STFT hyperparameters
SR = 22050
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB = 25          # Silence trimming threshold
TARGET_FRAMES = 128  # Target time frames (yields a 128x128 2D representation)

def extract_log_mel_spectrogram(
    file_path: str,
    sr: int = SR,
    n_fft: int = N_FFT,
    hop_length: int = HOP_LENGTH,
    n_mels: int = N_MELS,
    top_db: int = TOP_DB,
    target_frames: int = TARGET_FRAMES
) -> np.ndarray:
    """
    Loads raw wav audio, trims studio silence buffer, computes Log-Mel spectrogram,
    and standardizes time dimension to target_frames via center-cropping or padding.
    
    Returns:
        np.ndarray of shape (n_mels, target_frames), dtype float32.
    """
    # 1. Load audio waveform
    y, _ = librosa.load(file_path, sr=sr)
    
    # 2. Trim leading/trailing studio silence
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)
    
    # Fallback guard for empty or near-silent files
    if len(y_trimmed) < hop_length:
        y_trimmed = y
        
    # 3. Compute Mel Spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y=y_trimmed,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=2.0
    )
    
    # 4. Convert power to decibel scale (relative to peak)
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    
    # 5. Fix time axis to TARGET_FRAMES (pad or center-crop)
    n_frames = log_mel.shape[1]
    if n_frames < target_frames:
        pad_width = target_frames - n_frames
        log_mel_fixed = np.pad(
            log_mel,
            ((0, 0), (0, pad_width)),
            mode="constant",
            constant_values=log_mel.min()
        )
    elif n_frames > target_frames:
        start_idx = (n_frames - target_frames) // 2
        log_mel_fixed = log_mel[:, start_idx : start_idx + target_frames]
    else:
        log_mel_fixed = log_mel

    return log_mel_fixed.astype(np.float32)

In [10]:
# Audit a single trial
sample_path = df_cleaned["file_path"].iloc[0]
sample_spec = extract_log_mel_spectrogram(sample_path)

print(f"Sample file: {Path(sample_path).name}")
print(f"Output shape: {sample_spec.shape} (Expected: (128, 128))")
print(f"Dtype: {sample_spec.dtype} (Expected: float32)")
print(f"Dynamic range (dB): [{sample_spec.min():.2f}, {sample_spec.max():.2f}]")

Sample file: 03-01-01-01-01-01-01.wav
Output shape: (128, 128) (Expected: (128, 128))
Dtype: float32 (Expected: float32)
Dynamic range (dB): [-80.00, 0.00]
